In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set DEBUG = True for rapid code verification (seconds), False for full training.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "query_samples": 32,
    "candidate_samples": 64,
    "teacher_epochs": 2,
    "student_epochs": 2,
    "batch_size": 4,
    "log_interval": 2,
    "teacher_lr": 2e-4,
    "student_lr": 4e-4,
    "lorma_rank": 16,
    "lorma_alpha": 32,
    "kd_temperature": 0.05,
    "kd_alpha": 0.5,  # Weight for Distillation Loss vs Hard InfoNCE Loss
    "max_seq_len": 768
}

FULL_CONFIG = {
    "query_samples": None,  # Full dataset
    "candidate_samples": None,
    "teacher_epochs": 3,
    "student_epochs": 3,
    "batch_size": 8,
    "log_interval": 20,
    "teacher_lr": 2e-4,
    "student_lr": 4e-4,
    "lorma_rank": 16,
    "lorma_alpha": 32,
    "kd_temperature": 0.05,
    "kd_alpha": 0.5,
    "max_seq_len": 1024
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Phase 7 PCR Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Phase 7 PCR Execution Mode: PRODUCTION RUN ===
Configuration: {'query_samples': None, 'candidate_samples': None, 'teacher_epochs': 3, 'student_epochs': 3, 'batch_size': 8, 'log_interval': 20, 'teacher_lr': 0.0002, 'student_lr': 0.0004, 'lorma_rank': 16, 'lorma_alpha': 32, 'kd_temperature': 0.05, 'kd_alpha': 0.5, 'max_seq_len': 1024}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_TEACHER = "A-240242/legal-modern-decoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_pcr_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_7_PCR/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 7 Prior Case Retrieval Data...")
raw_train_queries = load_pcr_file("Train/train.json")
train_candidates = load_pcr_file("Train/train_candidates.json")

raw_dev_queries = load_pcr_file("Dev/dev.json")
dev_candidates = load_pcr_file("Dev/dev_candidates.json")

raw_test_queries = load_pcr_file("Test/test.json")
test_candidates = load_pcr_file("Test/test_candidates.json")

def parse_queries(raw_data):
    """
    Normalizes query data into a standardized list of dicts regardless of 
    whether the input JSON is a dictionary (id -> query_obj) or a list.
    """
    if isinstance(raw_data, dict):
        if "queries" in raw_data and isinstance(raw_data["queries"], list):
            return raw_data["queries"]
        parsed = []
        for q_id, q_val in raw_data.items():
            if isinstance(q_val, dict):
                item = dict(q_val)
                item["id"] = str(q_id)
                parsed.append(item)
            else:
                parsed.append({"id": str(q_id), "text": q_val})
        return parsed
    elif isinstance(raw_data, list):
        return raw_data
    return []

def format_case_text(text_field):
    if isinstance(text_field, list):
        return " ".join(text_field)
    return str(text_field)

# Normalize queries into standard lists
train_queries = parse_queries(raw_train_queries)
dev_queries = parse_queries(raw_dev_queries)
test_queries = parse_queries(raw_test_queries)

print(f"Raw Loaded Counts -> Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")

# Subsample if running in DEBUG mode
if DEBUG and CFG["query_samples"]:
    train_queries = train_queries[:CFG["query_samples"]]
    dev_queries = dev_queries[:CFG["query_samples"]]
    test_queries = test_queries[:CFG["query_samples"]]
    
    if isinstance(train_candidates, dict):
        train_cand_keys = list(train_candidates.keys())[:CFG["candidate_samples"]]
        train_candidates = {k: train_candidates[k] for k in train_cand_keys}
        
    if isinstance(dev_candidates, dict):
        dev_cand_keys = list(dev_candidates.keys())[:CFG["candidate_samples"]]
        dev_candidates = {k: dev_candidates[k] for k in dev_cand_keys}
        
    if isinstance(test_candidates, dict):
        test_cand_keys = list(test_candidates.keys())[:CFG["candidate_samples"]]
        test_candidates = {k: test_candidates[k] for k in test_cand_keys}
    
    print(f"\n[DEBUG] Subsampled -> Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Raw Loaded Counts -> Train Queries: 827, Dev Queries: 118, Test Queries: 237


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class PCRBiEncoderDataset(Dataset):
    """
    Dataset for Training Bi-Encoder Query-Candidate Pairs.
    Pairs each Query document with one of its positive relevant candidate documents.
    """
    def __init__(self, queries, candidates_dict, tokenizer, max_length=1024):
        self.samples = []
        
        for q in queries:
            q_id = q.get("id")
            q_text = format_case_text(q.get("text", ""))
            rel_cands = q.get("relevant_candidates", [])
            
            for c_id in rel_cands:
                if c_id in candidates_dict:
                    c_text = format_case_text(candidates_dict[c_id])
                    self.samples.append({
                        "query_id": q_id,
                        "query_text": q_text,
                        "candidate_id": c_id,
                        "candidate_text": c_text
                    })
                    
        # Fallback if no candidate matches found in debug slice
        if len(self.samples) == 0 and len(queries) > 0 and len(candidates_dict) > 0:
            first_c_id = list(candidates_dict.keys())[0]
            first_c_text = format_case_text(candidates_dict[first_c_id])
            for q in queries:
                self.samples.append({
                    "query_id": q.get("id"),
                    "query_text": format_case_text(q.get("text", "")),
                    "candidate_id": first_c_id,
                    "candidate_text": first_c_text
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def pcr_collate_fn(batch):
    query_texts = [item["query_text"] for item in batch]
    candidate_texts = [item["candidate_text"] for item in batch]
    
    q_tok = tokenizer(
        query_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt"
    )
    c_tok = tokenizer(
        candidate_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt"
    )
    
    return {
        "query_input_ids": q_tok["input_ids"],
        "query_attention_mask": q_tok["attention_mask"],
        "candidate_input_ids": c_tok["input_ids"],
        "candidate_attention_mask": c_tok["attention_mask"]
    }

print("Building Phase 7 Pair DataLoader...")
train_dataset = PCRBiEncoderDataset(train_queries, train_candidates, tokenizer, max_length=CFG["max_seq_len"])
train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=pcr_collate_fn)

print(f"Created {len(train_dataset)} training query-positive candidate pair samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building Phase 7 Pair DataLoader...
Created 5308 training query-positive candidate pair samples.


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class ModernDecoderAttention(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + attn_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class SwiGLUMLP(nn.Module):
    def __init__(self, hidden_size=768, intermediate_size=2048):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class ModernDecoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = RMSNorm(hidden_size)
        self.self_attn = ModernDecoderAttention(hidden_size, num_heads)
        self.post_attention_layernorm = RMSNorm(hidden_size)
        self.mlp = SwiGLUMLP(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin)
        x = residual + x
        
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x
        return x


class ModernDecoderBackbone(nn.Module):
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            ModernDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin)
            
        x = self.norm(x)
        return x


class CustomModernDecoderDenseEmbedder(nn.Module):
    """Modern Decoder Dense Embedder using Mean Pooling."""
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = ModernDecoderBackbone(vocab_size, hidden_size, num_layers, num_heads, intermediate_size)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        hidden_states = self.model(input_ids, attention_mask=attention_mask)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_hidden = (hidden_states * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            pooled = sum_hidden / sum_mask
        else:
            pooled = hidden_states.mean(dim=1)
            
        # L2 Normalize Embeddings
        return F.normalize(pooled, p=2, dim=-1)

print("Custom Modern Decoder Dense Retrieval Embedder written from scratch successfully.")

Custom Modern Decoder Dense Retrieval Embedder written from scratch successfully.


In [5]:
class LormaAdapter(nn.Module):
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.scaling * self.B(self.A(x))


class LoRMAWrappedLinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        self.adapter = LormaAdapter(
            base_linear.in_features, base_linear.out_features, rank=rank, alpha=alpha
        )

    def forward(self, x):
        return self.base(x) + self.adapter(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# 1. TEACHER MODEL (Full 6-Layer Model + Phase 1 Weights + LoRMA)
# ==============================================================================
teacher_model = CustomModernDecoderDenseEmbedder(
    vocab_size=len(tokenizer), 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

print("Fetching Phase 1 Modern Decoder Checkpoint Weights for Teacher...")
weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_TEACHER, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(weight_path)
teacher_model.load_state_dict(state_dict, strict=False)

# Freeze Teacher base weights and inject LoRMA adapters
for param in teacher_model.parameters():
    param.requires_grad = False

for layer in teacher_model.model.layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, LoRMAWrappedLinear(base_linear, rank=CFG["lorma_rank"], alpha=CFG["lorma_alpha"]))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, LoRMAWrappedLinear(base_linear, rank=CFG["lorma_rank"], alpha=CFG["lorma_alpha"]))

for name, param in teacher_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

teacher_model.to(device)

# ==============================================================================
# 2. STUDENT MODEL (Compact 3-Layer Model with ~50% parameters)
# ==============================================================================
student_model = CustomModernDecoderDenseEmbedder(
    vocab_size=len(tokenizer), 
    hidden_size=384,  # Scaled down hidden size
    num_layers=3,     # Scaled down layers
    num_heads=6, 
    intermediate_size=1024
).to(device)

teacher_trainable = sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)
student_total = sum(p.numel() for p in student_model.parameters())

print(f"\nArchitecture Summary:")
print(f"  - Teacher Model Total Parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
print(f"  - Teacher Trainable Parameters (LoRMA): {teacher_trainable:,}")
print(f"  - Student Model Total Parameters (Compact): {student_total:,}")

Fetching Phase 1 Modern Decoder Checkpoint Weights for Teacher...


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            


Architecture Summary:
  - Teacher Model Total Parameters: 68,454,144
  - Teacher Trainable Parameters (LoRMA): 1,400,832
  - Student Model Total Parameters (Compact): 17,599,104


In [6]:
@torch.no_grad()
def evaluate_pcr_model(eval_model, dev_queries, dev_candidates, tokenizer, device, top_k=7):
    """
    Evaluates Prior Case Retrieval quality using Micro-F1@7 metric[cite: 6].
    """
    eval_model.eval()
    cand_ids = list(dev_candidates.keys())
    cand_texts = [format_case_text(dev_candidates[cid]) for cid in cand_ids]
    
    cand_embeddings = []
    batch_size = 16
    for i in range(0, len(cand_texts), batch_size):
        batch_texts = cand_texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        emb = eval_model(tok["input_ids"], attention_mask=tok["attention_mask"])
        cand_embeddings.append(emb.cpu())
        
    if len(cand_embeddings) == 0:
        return 0.0, 0.0, 0.0
        
    cand_embeddings = torch.cat(cand_embeddings, dim=0) # [N_cand, d]
    
    total_tp = 0
    total_pred = 0
    total_gold = 0
    
    for q in dev_queries:
        q_text = format_case_text(q.get("text", ""))
        gold_cands = set(q.get("relevant_candidates", []))
        
        tok = tokenizer(q_text, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        q_emb = eval_model(tok["input_ids"], attention_mask=tok["attention_mask"]).cpu()
        
        sim_scores = torch.matmul(q_emb, cand_embeddings.T).squeeze(0)
        top_indices = torch.topk(sim_scores, k=min(top_k, len(cand_ids))).indices.tolist()
        
        retrieved_ids = [cand_ids[idx] for idx in top_indices]
        
        tp = len(set(retrieved_ids).intersection(gold_cands))
        total_tp += tp
        total_pred += len(retrieved_ids)
        total_gold += len(gold_cands)
        
    precision = total_tp / max(1, total_pred)
    recall = total_tp / max(1, total_gold)
    micro_f1 = (2 * precision * recall) / max(1e-9, precision + recall)
    
    return micro_f1, precision, recall


TEACHER_CHECKPOINT_DIR = "./checkpoints_phase7_teacher_lorma"
os.makedirs(TEACHER_CHECKPOINT_DIR, exist_ok=True)
TEACHER_CHECKPOINT_PATH = os.path.join(TEACHER_CHECKPOINT_DIR, "best_teacher_modern_lorma.pt")

teacher_epochs = CFG["teacher_epochs"]
teacher_optimizer = torch.optim.AdamW([p for p in teacher_model.parameters() if p.requires_grad], lr=CFG["teacher_lr"], weight_decay=0.01)

best_teacher_f1 = -1.0

print(f"\n=== TASK 1: Training Teacher Model with LoRMA ({teacher_epochs} Epochs) ===")

for epoch in range(teacher_epochs):
    teacher_model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        teacher_optimizer.zero_grad()
        
        q_ids = batch["query_input_ids"].to(device)
        q_mask = batch["query_attention_mask"].to(device)
        c_ids = batch["candidate_input_ids"].to(device)
        c_mask = batch["candidate_attention_mask"].to(device)
        
        q_embeds = teacher_model(q_ids, attention_mask=q_mask)
        c_embeds = teacher_model(c_ids, attention_mask=c_mask)
        
        logits = torch.matmul(q_embeds, c_embeds.T) / CFG["kd_temperature"]
        labels = torch.arange(logits.size(0), device=device, dtype=torch.long)
        
        loss = F.cross_entropy(logits, labels)
        
        loss.backward()
        teacher_optimizer.step()
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Teacher | Epoch {epoch+1}/{teacher_epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / max(1, len(train_loader))
    
    dev_f1, dev_p, dev_r = evaluate_pcr_model(teacher_model, dev_queries, dev_candidates, tokenizer, device)
    print(f"Teacher | Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Dev Micro-F1@7: {dev_f1:.4f} (P@7: {dev_p:.4f}, R@7: {dev_r:.4f})")

    if dev_f1 > best_teacher_f1 or epoch == 0:
        best_teacher_f1 = dev_f1
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in teacher_model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_micro_f1": best_teacher_f1,
            "trainable_state_dict": trainable_state_dict
        }
        torch.save(checkpoint_data, TEACHER_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Teacher Checkpoint (Dev Micro-F1@7: {best_teacher_f1:.4f})")
        
    print("-" * 75)


=== TASK 1: Training Teacher Model with LoRMA (3 Epochs) ===
Teacher | Epoch 1/3 | Step 20/664 | Loss: 1.5743
Teacher | Epoch 1/3 | Step 40/664 | Loss: 1.9620
Teacher | Epoch 1/3 | Step 60/664 | Loss: 1.5322
Teacher | Epoch 1/3 | Step 80/664 | Loss: 1.6129
Teacher | Epoch 1/3 | Step 100/664 | Loss: 1.8211
Teacher | Epoch 1/3 | Step 120/664 | Loss: 2.0034
Teacher | Epoch 1/3 | Step 140/664 | Loss: 1.2272
Teacher | Epoch 1/3 | Step 160/664 | Loss: 1.6050
Teacher | Epoch 1/3 | Step 180/664 | Loss: 1.6749
Teacher | Epoch 1/3 | Step 200/664 | Loss: 1.6234
Teacher | Epoch 1/3 | Step 220/664 | Loss: 1.6382
Teacher | Epoch 1/3 | Step 240/664 | Loss: 1.7342
Teacher | Epoch 1/3 | Step 260/664 | Loss: 1.3020
Teacher | Epoch 1/3 | Step 280/664 | Loss: 2.0768
Teacher | Epoch 1/3 | Step 300/664 | Loss: 1.2290
Teacher | Epoch 1/3 | Step 320/664 | Loss: 0.7629
Teacher | Epoch 1/3 | Step 340/664 | Loss: 2.0403
Teacher | Epoch 1/3 | Step 360/664 | Loss: 0.8971
Teacher | Epoch 1/3 | Step 380/664 | Loss:

In [7]:
# ==============================================================================
# TEACHER MODEL INFERENCE & SUBMISSION GENERATION
# ==============================================================================
print("Reloading Best Teacher Checkpoint for Phase 7 Test Retrieval...")

if os.path.exists(TEACHER_CHECKPOINT_PATH):
    checkpoint = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)
    teacher_model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded Teacher Checkpoint (Dev Micro-F1@7: {checkpoint['best_micro_f1']:.4f})\n")

teacher_model.eval()

# Encode Test Candidates with Teacher Model
test_cand_ids = list(test_candidates.keys())
test_cand_texts = [format_case_text(test_candidates[cid]) for cid in test_cand_ids]

print(f"Teacher Model: Encoding {len(test_cand_ids)} Test Candidate Documents...")

cand_embeddings_list_t = []
batch_size = 16

with torch.no_grad():
    for i in range(0, len(test_cand_texts), batch_size):
        batch_texts = test_cand_texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        emb = teacher_model(tok["input_ids"], attention_mask=tok["attention_mask"])
        cand_embeddings_list_t.append(emb.cpu())

test_cand_matrix_t = torch.cat(cand_embeddings_list_t, dim=0)

# Rank Candidates for Test Queries
print(f"Teacher Model: Ranking Candidates for {len(test_queries)} Test Queries...")
predictions_teacher = {}

for q in test_queries:
    q_id = str(q.get("id"))
    q_text = format_case_text(q.get("text", ""))
    
    with torch.no_grad():
        tok = tokenizer(q_text, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        q_emb = teacher_model(tok["input_ids"], attention_mask=tok["attention_mask"]).cpu()
        
    scores = torch.matmul(q_emb, test_cand_matrix_t.T).squeeze(0)
    top_k = min(7, len(test_cand_ids))
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    
    retrieved_cands = [str(test_cand_ids[idx]) for idx in top_indices]
    
    while len(retrieved_cands) < 7:
        for cid in test_cand_ids:
            cid_str = str(cid)
            if cid_str not in retrieved_cands:
                retrieved_cands.append(cid_str)
            if len(retrieved_cands) == 7:
                break
                
    predictions_teacher[q_id] = retrieved_cands[:7]

# Save Teacher Submission Zip
output_dir_t = "./phase7_teacher_lorma_pcr"
os.makedirs(output_dir_t, exist_ok=True)
pred_path_t = os.path.join(output_dir_t, "prediction.json")
sub_zip_path_t = os.path.join(output_dir_t, "submission.zip")

with open(pred_path_t, "w", encoding="utf-8") as f:
    json.dump(predictions_teacher, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path_t, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path_t, arcname="prediction.json")

print(f"\nPhase 7 Teacher PCR Prediction Complete! Generated rankings for {len(predictions_teacher)} queries[cite: 6].")
print(f"Teacher Submission saved to: {sub_zip_path_t}")

if DEBUG:
    print("\n[DEBUG] Sample Teacher Prediction Output:")
    sample_keys = list(predictions_teacher.keys())[:2]
    print(json.dumps({k: predictions_teacher[k] for k in sample_keys}, indent=2))

Reloading Best Teacher Checkpoint for Phase 7 Test Retrieval...
Successfully reloaded Teacher Checkpoint (Dev Micro-F1@7: 0.1812)

Teacher Model: Encoding 1727 Test Candidate Documents...
Teacher Model: Ranking Candidates for 237 Test Queries...

Phase 7 Teacher PCR Prediction Complete! Generated rankings for 237 queries[cite: 6].
Teacher Submission saved to: ./phase7_teacher_lorma_pcr/submission.zip


In [8]:
# Load Best Teacher Model for Distillation
if os.path.exists(TEACHER_CHECKPOINT_PATH):
    checkpoint = torch.load(TEACHER_CHECKPOINT_PATH, map_location=device)
    teacher_model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print("Loaded Best Teacher Model for Knowledge Distillation.\n")

teacher_model.eval()

STUDENT_CHECKPOINT_DIR = "./checkpoints_phase7_student_kd"
os.makedirs(STUDENT_CHECKPOINT_DIR, exist_ok=True)
STUDENT_CHECKPOINT_PATH = os.path.join(STUDENT_CHECKPOINT_DIR, "best_student_kd.pt")

student_epochs = CFG["student_epochs"]
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=CFG["student_lr"], weight_decay=0.01)

alpha = CFG["kd_alpha"]
best_student_f1 = -1.0

print(f"=== TASK 2: Knowledge Distillation to Student Model ({student_epochs} Epochs) ===")

for epoch in range(student_epochs):
    student_model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        student_optimizer.zero_grad()
        
        q_ids = batch["query_input_ids"].to(device)
        q_mask = batch["query_attention_mask"].to(device)
        c_ids = batch["candidate_input_ids"].to(device)
        c_mask = batch["candidate_attention_mask"].to(device)
        
        # 1. Forward pass through frozen Teacher
        with torch.no_grad():
            q_emb_t = teacher_model(q_ids, attention_mask=q_mask)
            c_emb_t = teacher_model(c_ids, attention_mask=c_mask)
            teacher_logits = torch.matmul(q_emb_t, c_emb_t.T) / CFG["kd_temperature"]
            
        # 2. Forward pass through Student
        q_emb_s = student_model(q_ids, attention_mask=q_mask)
        c_emb_s = student_model(c_ids, attention_mask=c_mask)
        student_logits = torch.matmul(q_emb_s, c_emb_s.T) / CFG["kd_temperature"]
        
        # 3. Compute Soft KD Loss (MSE on Similarity Logits) + Hard InfoNCE Loss
        labels = torch.arange(student_logits.size(0), device=device, dtype=torch.long)
        hard_loss = F.cross_entropy(student_logits, labels)
        soft_loss = F.mse_loss(student_logits, teacher_logits)
        
        total_loss = (1.0 - alpha) * hard_loss + alpha * soft_loss
        
        total_loss.backward()
        student_optimizer.step()
        total_train_loss += total_loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Student KD | Epoch {epoch+1}/{student_epochs} | Step {step+1}/{len(train_loader)} | Total Loss: {total_loss.item():.4f} (Soft MSE: {soft_loss.item():.4f})")
            
    avg_train_loss = total_train_loss / max(1, len(train_loader))
    
    dev_f1, dev_p, dev_r = evaluate_pcr_model(student_model, dev_queries, dev_candidates, tokenizer, device)
    print(f"Student KD | Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Dev Micro-F1@7: {dev_f1:.4f} (P@7: {dev_p:.4f}, R@7: {dev_r:.4f})")

    if dev_f1 > best_student_f1 or epoch == 0:
        best_student_f1 = dev_f1
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_micro_f1": best_student_f1,
            "state_dict": student_model.state_dict()
        }
        torch.save(checkpoint_data, STUDENT_CHECKPOINT_PATH)
        print(f"  [+] Saved Best Distilled Student Checkpoint (Dev Micro-F1@7: {best_student_f1:.4f})")
        
    print("-" * 75)

Loaded Best Teacher Model for Knowledge Distillation.

=== TASK 2: Knowledge Distillation to Student Model (3 Epochs) ===
Student KD | Epoch 1/3 | Step 20/664 | Total Loss: 6.0107 (Soft MSE: 9.6641)
Student KD | Epoch 1/3 | Step 40/664 | Total Loss: 6.2273 (Soft MSE: 10.4663)
Student KD | Epoch 1/3 | Step 60/664 | Total Loss: 3.2657 (Soft MSE: 4.9929)
Student KD | Epoch 1/3 | Step 80/664 | Total Loss: 2.5847 (Soft MSE: 3.9866)
Student KD | Epoch 1/3 | Step 100/664 | Total Loss: 2.6538 (Soft MSE: 4.2928)
Student KD | Epoch 1/3 | Step 120/664 | Total Loss: 3.6449 (Soft MSE: 5.9979)
Student KD | Epoch 1/3 | Step 140/664 | Total Loss: 2.8329 (Soft MSE: 4.8922)
Student KD | Epoch 1/3 | Step 160/664 | Total Loss: 4.7376 (Soft MSE: 7.1635)
Student KD | Epoch 1/3 | Step 180/664 | Total Loss: 1.8226 (Soft MSE: 2.9623)
Student KD | Epoch 1/3 | Step 200/664 | Total Loss: 2.0580 (Soft MSE: 3.0640)
Student KD | Epoch 1/3 | Step 220/664 | Total Loss: 2.0052 (Soft MSE: 2.6680)
Student KD | Epoch 1/3 

In [9]:
# ==============================================================================
# STUDENT MODEL INFERENCE & SUBMISSION GENERATION
# ==============================================================================
print("Reloading Best Distilled Student Checkpoint for Phase 7 Test Retrieval...")

if os.path.exists(STUDENT_CHECKPOINT_PATH):
    checkpoint = torch.load(STUDENT_CHECKPOINT_PATH, map_location=device)
    student_model.load_state_dict(checkpoint["state_dict"])
    print(f"Successfully reloaded Distilled Student Checkpoint (Dev Micro-F1@7: {checkpoint['best_micro_f1']:.4f})\n")

student_model.eval()

# Encode Test Candidates with Student Model
print(f"Student Model: Encoding {len(test_cand_ids)} Test Candidate Documents...")

cand_embeddings_list_s = []
batch_size = 16

with torch.no_grad():
    for i in range(0, len(test_cand_texts), batch_size):
        batch_texts = test_cand_texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        emb = student_model(tok["input_ids"], attention_mask=tok["attention_mask"])
        cand_embeddings_list_s.append(emb.cpu())

test_cand_matrix_s = torch.cat(cand_embeddings_list_s, dim=0)

# Rank Candidates for Test Queries
print(f"Student Model: Ranking Candidates for {len(test_queries)} Test Queries...")
predictions_student = {}

for q in test_queries:
    q_id = str(q.get("id"))
    q_text = format_case_text(q.get("text", ""))
    
    with torch.no_grad():
        tok = tokenizer(q_text, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        q_emb = student_model(tok["input_ids"], attention_mask=tok["attention_mask"]).cpu()
        
    scores = torch.matmul(q_emb, test_cand_matrix_s.T).squeeze(0)
    top_k = min(7, len(test_cand_ids))
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    
    retrieved_cands = [str(test_cand_ids[idx]) for idx in top_indices]
    
    while len(retrieved_cands) < 7:
        for cid in test_cand_ids:
            cid_str = str(cid)
            if cid_str not in retrieved_cands:
                retrieved_cands.append(cid_str)
            if len(retrieved_cands) == 7:
                break
                
    predictions_student[q_id] = retrieved_cands[:7]

# Save Student Submission Zip
output_dir_s = "./phase7_student_kd_pcr"
os.makedirs(output_dir_s, exist_ok=True)
pred_path_s = os.path.join(output_dir_s, "prediction.json")
sub_zip_path_s = os.path.join(output_dir_s, "submission.zip")

with open(pred_path_s, "w", encoding="utf-8") as f:
    json.dump(predictions_student, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path_s, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path_s, arcname="prediction.json")

print(f"\nPhase 7 Distilled Student PCR Prediction Complete! Generated rankings for {len(predictions_student)} queries[cite: 6].")
print(f"Student Submission saved to: {sub_zip_path_s}")

if DEBUG:
    print("\n[DEBUG] Sample Student Prediction Output:")
    sample_keys = list(predictions_student.keys())[:2]
    print(json.dumps({k: predictions_student[k] for k in sample_keys}, indent=2))

Reloading Best Distilled Student Checkpoint for Phase 7 Test Retrieval...
Successfully reloaded Distilled Student Checkpoint (Dev Micro-F1@7: 0.1823)

Student Model: Encoding 1727 Test Candidate Documents...
Student Model: Ranking Candidates for 237 Test Queries...

Phase 7 Distilled Student PCR Prediction Complete! Generated rankings for 237 queries[cite: 6].
Student Submission saved to: ./phase7_student_kd_pcr/submission.zip
